## Complements

Run a qualitative sanity check for chosen items: 
Coke Classic  (ID: 16696)
Banana  (ID: 24852)
Eggo Homestyle Waffles  (ID: 30696)

In [14]:
import pandas as pd
%run ../utils/complements.py
%run ../utils/results.py

file_path_root = "../data/validation/complements/"

# Load files
product_df = pd.read_csv('../data/cleaned/product-info-full.csv')
orders_full_df = pd.read_csv('../dataset/order_products__prior.csv')

focus_products = [16696,24852,30696]

dept1_df = pd.read_csv('../data/validation/pairwise/pairwise-dept1.csv')
dept7_df = pd.read_csv('../data/validation/pairwise/pairwise-dept7.csv')
dept4_df = pd.read_csv('../data/validation/pairwise/pairwise-dept4.csv')

pairwise_df = pd.concat([dept1_df, dept4_df, dept7_df], ignore_index=True)

num_orders, min_pij = get_min_pij()

lift_df = compute_lift(focus_products, pairwise_df, min_pij=min_pij, total_orders=num_orders)
complements_df = compute_hybrid_score(lift_df, focus_products, top_n=10)
network_df = compute_network_enhanced_impact(complements_df, pairwise_df)
network_df.to_csv(f"{file_path_root}minimal-complements.csv", index=False)

show_comp_results(network_df)

Computing lift: 100%|██████████| 3/3 [00:05<00:00,  1.70s/it]



Product: Coke Classic  (ID: 16696)


,comp_name,impact_index,hybrid_score,aisle
0,Lemon Lime Soda Caffeine Free,1.000000,1.000000,soft drinks
1,Deluxe Mixed Nuts,0.729776,0.423193,nuts seeds dried fruit
2,Miniatures Assortment Party Bag,0.587203,0.393006,candy chocolate
3,Classic Caffeine Free Soda,0.558162,0.839998,soft drinks
4,Vanilla Coke,0.458006,0.671695,soft drinks
5,Ginger Soda,0.449769,0.599240,soft drinks
6,Multi-Grain English Muffins,0.368021,0.455654,breakfast bakery
7,Chocolate Favorites Fun Size Variety Pack,0.350483,0.533326,candy chocolate
8,Seasoned Black Cherry Barbecue Pork jerky,0.290181,0.508171,popcorn jerky
9,Roasted Garlic Hummus with Pretzels,0.226085,0.407265,fresh dips tapenades



Product: Banana  (ID: 24852)


,comp_name,impact_index,hybrid_score,aisle
0,Disney Frozen Kids Yogurt,0.021073,0.032329,yogurt
1,Shells & White Cheddar Mac & Cheese Family Siz...,0.020555,0.029361,instant foods
2,Eggs,0.020220,0.028862,eggs
3,Reduced Fat Shredded Mozzarella Cheese,0.020045,0.027501,packaged cheese
4,2nd Foods Organic Pear and Spinach Baby Food,0.019898,0.030188,baby food formula
5,"Veg and Fruit Puree, 100%, Organic, Sweet Pota...",0.019890,0.029024,baby food formula
6,Humm! Cocktail Hummus Roasted Pine Nuts,0.019383,0.027550,fresh dips tapenades
7,Trop50 Some Pulp Orange Juice,0.019214,0.027036,refrigerated
8,Cashew & Ginger Spice Fruit & Nut Bar,0.018888,0.027920,energy granola bars
9,Slim Cut Reduced Fat 2% Milk Sharp Cheddar Cheese,0.018882,0.026842,packaged cheese



Product: Eggo Homestyle Waffles  (ID: 30696)


,comp_name,impact_index,hybrid_score,aisle
0,Mini Pancakes,0.982143,0.574832,frozen breakfast
1,Original Lite Syrup,0.749872,0.537448,honeys syrups nectars
2,Natural Butter Flavor Lite Maple Syrup,0.619897,0.680947,honeys syrups nectars
3,Original Thin Sausage Pizza,0.554231,0.819532,frozen pizza
4,Cinnamon French Toaster Sticks,0.534244,0.516950,frozen breakfast
5,Original Chicken Breast Nuggets,0.454190,0.521964,packaged poultry
6,Butter Rich Maple Syrup,0.448560,0.652078,honeys syrups nectars
7,Original Patties (100965) 12 Oz Breakfast,0.399133,0.576183,hot dogs bacon sausage
8,Natural Low Moisture Part Skim Mozzarella Chee...,0.374672,0.534710,packaged cheese
9,Mild Taco Seasoning Mix,0.341598,0.504242,marinades meat preparation


Sample 5000 products, split into train and test, and calculate pairwise probabilities.

In [15]:
%run ../utils/pairwise.py
%run ../utils/sampling.py
%run ../utils/complements.py
import pandas as pd

file_path_root = "../data/validation/complements/"

# Load files
all_orders_df = pd.read_csv('../data/cleaned/order-products-full.csv')
products_df = pd.read_csv('../data/cleaned/product-info-full.csv')

file_path_base = "../data/validation/complements/"

# Get list of sampled products and split orders into train and test sets
sampled_products, train_df, test_df = sample_products_and_split_orders(
    products_df,
    all_orders_df,
    target_sample_size=5000,
    min_orders=50,
    test_size=0.2,
    random_state=42
)

# Simplify train orders df
order_product_df = train_df[['order_id', 'product_id']]

# Compute probabilities for the train set
product_pair_df = compute_pairwise_probabilities_sample(order_product_df, 
    sampled_products,
    output_csv=f"{file_path_base}train-pairwise.csv",
    batch_size=1000
)

# Simplify test orders df
order_product_test_df = test_df[['order_id', 'product_id']]

# Compute probabilities for the test set
product_pair_test_df = compute_pairwise_probabilities_sample(order_product_test_df,
    sampled_products,
    output_csv=f"{file_path_base}test-pairwise.csv",
    batch_size=1000
)

Sampled 5000 products from 26686 eligible products.
Train orders: 5049729 rows, Test orders: 1264861 rows


100%|██████████| 5/5 [00:32<00:00,  6.43s/it]


Completed computation. Saved to ../data/validation/complements/train-pairwise.csv


100%|██████████| 5/5 [00:16<00:00,  3.21s/it]

Completed computation. Saved to ../data/validation/complements/test-pairwise.csv


Run complement pipeline validation on the 5000-product sample.

In [18]:
%run ../utils/complements.py
import pandas as pd


# Load dataframes
pairwise_train_df = pd.read_csv(f"{file_path_base}train-pairwise.csv")
pairwise_test_df = pd.read_csv(f"{file_path_base}test-pairwise.csv")
orders_test_df = order_product_test_df.copy()

num_orders, min_pij = get_min_pij()

results = run_samples_validation(
    pairwise_train_df,
    pairwise_test_df,
    all_orders_df,
    sampled_products,
    orders_test_df,
    num_orders,
    min_pij,
    10
)


--- Sample 1/10 (size=100) ---
Complement pipeline time: 3.5s, rows=530


100%|██████████| 1/1 [00:04<00:00,  4.06s/it]


Completed computation. Saved to DataFrame


100%|██████████| 1/1 [00:03<00:00,  3.65s/it]


Completed computation. Saved to DataFrame


...

Completed computation. Saved to DataFrame


100%|██████████| 1/1 [00:03<00:00,  3.14s/it]


Completed computation. Saved to DataFrame


100%|██████████| 1/1 [00:03<00:00,  3.11s/it]


Completed computation. Saved to DataFrame
Validation (sample 10) summary: {'precision@N': 0.8, 'recall@N': 0.020023604775864633, 'hit_rate': 0.9777777777777777, 'coverage_fraction': 1.0}

Temporal metrics across samples (mean ± std):
       precision@N  recall@N  hit_rate  coverage_fraction
mean     0.811560  0.018563  0.997778                1.0
std      0.030871  0.001709  0.007027                0.0

        COMPLEMENTS PIPELINE – SUMMARY         

--- Runtime Performance ---
Avg complements pipeline time:    2.14s  (std=0.47)

--- Temporal Metrics (Mean ± Std) ---
precision@N         : 0.8116 ± 0.0309
recall@N            : 0.0186 ± 0.0017
hit_rate            : 0.9978 ± 0.0070
coverage_fraction   : 1.0000 ± 0.0000

--- Complement Impact Summary (Top 10 Risky Products) ---
    removed_product  avg_total_impact  avg_neighbor_CII_before  \
2              2326          0.136451                      0.0   
29            21417          0.135928                      0.0   
7              3

Save results to clipboard for creating tables.

In [19]:
# ----  Temporal Metrics Table (mean ± std) ----
temporal_df = results['temporal_stats'].reset_index()
temporal_df.rename(columns={'index':'Statistic'}, inplace=True)
temporal_df = temporal_df.melt(id_vars='Statistic', var_name='Metric', value_name='Value')
temporal_df['Value'] = temporal_df['Value'].round(4)

temporal_df.to_clipboard(index=False)


In [20]:
# ----  Per-sample Temporal Metrics ----
per_sample_list = []
for idx, sample in enumerate(results['per_sample']):
    temporal = sample['temporal']
    temporal['sample'] = idx + 1
    per_sample_list.append(temporal)

per_sample_df = pd.DataFrame(per_sample_list).round(4)

per_sample_df.to_clipboard(index=False)


In [15]:
# ---- Delisting / Complement Impact Table ----
delist_df = results['delist_summary'].copy()
delist_df = delist_df[['removed_product','avg_total_impact','std_total_impact',
                           'avg_neighbor_CII_before','std_neighbor_CII_before','effectiveness_score']]
delist_df = delist_df.round(4)
delist_df = delist_df.sort_values('effectiveness_score', ascending=False).reset_index(drop=True)

delist_df.to_clipboard(index=False)